# SAT → Set Cover → PDNAR

Этот ноутбук делает три вещи:

1. **Сводит SAT к задаче минимального покрытия множествами (Set Cover)** —
   стандартная полиномиальная редукция (Karp-style).
2. **Обучает PDNAR** (*Primal-Dual Neural Algorithmic Reasoning*) — графовую нейросеть
   из репозитория [`pdnar`](https://github.com/artlvruran/pdnar), которая учится
   имитировать взвешенный primal-dual алгоритм для Set Cover.
3. **Делает инференс на одном SAT-примере**: строит из формулы граф, прогоняет через
   **обученную нейросеть** (а не классический алгоритм!) и по предсказанному покрытию
   восстанавливает присваивание переменных.

> ⚠️ **Важно.** Ноутбук использует **саму нейросеть PDNAR** (`GraphNeuralExecutor` +
> `BipartiteMPNN` из `src/model/`) для решения. Классический primal-dual алгоритм
> (`src/dataset/algorithms/set_cover.py`) в этом репозитории используется **только
> для генерации обучающих траекторий (hints)** — так же, как это сделано у авторов
> репозитория. На этапе инференса (раздел 5) этот классический алгоритм **не
> вызывается ни разу** — ответ строит исключительно обученная сеть.

### Как запускать

Положите этот ноутбук в корень склонированного репозитория `pdnar` (туда же, где
лежит `main.py`) и запускайте в conda-окружении из `environment.yml` (там уже есть
`torch`, `torch_geometric`, `pytorch-scatter`, `lightning`, `networkx`, `scipy`).


In [ ]:
import os
import sys
import random
from pathlib import Path

import numpy as np

# Clone the repository if it doesn't exist
repo_name = "pdnar"
if not Path(repo_name).exists():
    print(f"Cloning {repo_name} repository...")
    !git clone https://github.com/artlvruran/pdnar
    # Change current working directory to the cloned repository
    os.chdir(repo_name)
    print(f"Changed directory to {os.getcwd()}")

def find_repo_root(start="."):
    # Ищем корень репозитория pdnar (там, где лежит src/model/graph_executor.py).
    p = Path(start).resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "src" / "model" / "graph_executor.py").exists():
            return candidate
    raise FileNotFoundError(
        "Не найден корень репозитория pdnar. Поместите этот ноутбук в корень "
        "склонированного репозитория https://github.com/artlvruran/pdnar "
        "(там же, где лежит main.py)."
    )


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))
print("Корень репозитория PDNAR:", REPO_ROOT)

SEED = 0
random.seed(SEED)
np.random.seed(SEED)


Cloning pdnar repository...
Cloning into 'pdnar'...
remote: Enumerating objects: 113, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 113 (delta 43), reused 79 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (113/113), 2.90 MiB | 4.91 MiB/s, done.
Resolving deltas: 100% (43/43), done.
Changed directory to /content/pdnar
Корень репозитория PDNAR: /content/pdnar


In [ ]:
!pip install torch_geometric lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 40.6 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
from types import SimpleNamespace
from tqdm.auto import tqdm

import lightning as L
from torch_geometric.loader import DataLoader
from torch_geometric.utils import degree

# --- модули из репозитория pdnar ---
from src.dataset.data import BipartiteData
from src.dataset.graph import generate_bipartite_graphs
from src.dataset.algorithms.set_cover import set_cover as generate_set_cover_trace
from src.dataset.algorithms.set_cover_solver import solve_minimum_set_cover
from src.model.graph_executor import GraphNeuralExecutor   # <-- это и есть PDNAR
from src.model.lightning import VCEPLightning

torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cpu


## 1. Что такое PDNAR (кратко)

PDNAR представляет экземпляр Set Cover как **двудольный граф**:

* узлы **X** (RHS) — кандидатные множества с весами `w` (это то, что мы *выбираем*);
* узлы **Y** (LHS) — элементы вселенной, которые нужно покрыть (это то, на что
  накладываются ограничения покрытия).

Нейросеть (`GraphNeuralExecutor`, процессор — `BipartiteMPNN`) на каждом шаге:

1. кодирует текущие веса `x` энкодером,
2. прогоняет сообщения `X → Y → X` через `BipartiteMPNN` (это выученная, а не
   аналитическая версия primal-dual обновления двойственных переменных),
3. декодирует вероятность «удалить/выбрать» каждый узел `X` (стать частью покрытия)
   и новое значение остаточного веса,
4. помечает покрытые элементы `Y` и повторяет, пока вселенная не покрыта.

При обучении используется **teacher forcing** по траекториям классического
взвешенного primal-dual алгоритма (функция `set_cover(...)` в
`src/dataset/algorithms/set_cover.py`) — сеть учится предсказывать те же
шаги (удаление узла, новый остаточный вес, значение по рёбрам), что и алгоритм,
а также итоговое оптимальное покрытие (`primal_optimal_solution`, посчитанное
точно через MILP в `set_cover_solver.py`). На инференсе (тест-режим,
`GraphNeuralExecutor.test(...)`) сеть работает **автономно, авторегрессионно**,
без доступа к алгоритму-учителю — именно это мы и используем ниже для SAT.


## 2. Редукция SAT → Set Cover

Пусть КНФ-формула $\varphi$ имеет переменные $x_1,\dots,x_n$ и дизъюнкты
$C_1,\dots,C_m$.

**Вселенная** (элементы, которые нужно покрыть):

$$U=\{v_1,\dots,v_n\}\ \cup\ \{c_1,\dots,c_m\}$$

— по одному «переменному» элементу $v_i$ на переменную и по одному «дизъюнктному»
элементу $c_j$ на дизъюнкт.

**Кандидатные множества** — по два литеральных множества на переменную:

$$S_i^{+}=\{v_i\}\cup\{c_j : x_i\text{ входит в } C_j\text{ положительно}\}$$
$$S_i^{-}=\{v_i\}\cup\{c_j : x_i\text{ входит в } C_j\text{ отрицательно}\}$$

Веса всех множеств равны 1 (невзвешенный Set Cover = минимизация числа выбранных
множеств).

**Утверждение.** Минимальное покрытие $U$ множествами из $\{S_i^{\pm}\}$ имеет
размер **ровно $n$** тогда и только тогда, когда $\varphi$ **выполнима**.

*Доказательство.*
($\Leftarrow$) Если есть выполняющее присваивание $a$, возьмём $S_i^{+}$, если
$a(x_i)=\text{True}$, иначе $S_i^{-}$. Все $v_i$ покрыты по построению; каждый $c_j$
покрыт, т.к. в $C_j$ есть литерал, истинный при $a$, — а значит, соответствующее
$S_i^{\pm}$ выбрано и содержит $c_j$. Итого — покрытие размера $n$.

($\Rightarrow$) Пусть покрытие размера $n$ существует. Каждый элемент $v_i$
содержится только в $S_i^{+}$ и $S_i^{-}$, элементов $v_i$ ровно $n$, и множеств
выбрано тоже ровно $n$ — по принципу Дирихле на каждую переменную приходится
**ровно одно** выбранное множество (иначе для какой-то переменной было бы выбрано
0 множеств, и $v_i$ остался бы непокрыт). Значит, выбор задаёт корректное полное
присваивание $a$. Поскольку все $c_j$ покрыты, в каждом дизъюнкте есть литерал,
истинный при $a$ — значит, $a$ выполняет $\varphi$. $\blacksquare$

Ниже — код редукции и (только для проверки корректности самой редукции, полным
перебором на маленьких случайных примерах) её валидация.


In [ ]:
def sat_to_bipartite_data(clauses, n_vars, timesteps=None):
    # Строит BipartiteData (формат PDNAR) для Set Cover, полученного из SAT.
    #
    # Индексация X-узлов (кандидатных множеств): 2*i -> S_i^+, 2*i+1 -> S_i^-.
    # Индексация Y-узлов (элементов вселенной): [0, n_vars) -> v_i, [n_vars, n_vars+m) -> c_j.
    # edge_index[0] -- индекс Y (элемент), edge_index[1] -- индекс X (множество),
    # как в src/dataset/algorithms/set_cover.py.
    n_clauses = len(clauses)
    num_x = 2 * n_vars
    num_y = n_vars + n_clauses

    edges_y, edges_x = [], []
    for i in range(n_vars):
        edges_y += [i, i]
        edges_x += [2 * i, 2 * i + 1]
    for j, clause in enumerate(clauses):
        y = n_vars + j
        for lit in clause:
            var = abs(lit) - 1
            x = 2 * var if lit > 0 else 2 * var + 1
            edges_y.append(y)
            edges_x.append(x)

    edge_index = torch.tensor([edges_y, edges_x], dtype=torch.long)

    # невзвешенный Set Cover -> все веса равны
    weight = torch.ones(num_x, dtype=torch.float)
    T = timesteps or (num_x + 2)  # запас шагов; реально используется только x[:, 0]
    x = weight.view(-1, 1, 1).repeat(1, T, 1)   # (num_x, T, 1)
    y = torch.zeros(num_y, T, 1)                # значения y на инференсе не используются

    data = BipartiteData(x=x, y=y, edge_index=edge_index)
    meta = {"n_vars": n_vars, "n_clauses": n_clauses, "num_x": num_x, "num_y": num_y}
    return data, meta


def decode_assignment(pred_set, n_vars):
    # Восстанавливает присваивание x_1..x_n из предсказанного покрытия.
    assignment = [None] * n_vars
    ambiguous, uncovered = [], []
    for i in range(n_vars):
        plus = bool(pred_set[2 * i].item())
        minus = bool(pred_set[2 * i + 1].item())
        if plus and not minus:
            assignment[i] = True
        elif minus and not plus:
            assignment[i] = False
        elif plus and minus:
            assignment[i] = True  # оба литерала выбраны -- берём любой (переменная всё равно покрыта)
            ambiguous.append(i + 1)
        else:
            uncovered.append(i + 1)  # не должно происходить -- PDNAR гарантирует валидное покрытие
            assignment[i] = True
    return assignment, ambiguous, uncovered


def evaluate_cnf(clauses, assignment):
    # Проверка присваивания (НЕ решение SAT!) -- удовлетворяет ли оно всем дизъюнктам.
    for clause in clauses:
        if not any((lit > 0) == assignment[abs(lit) - 1] for lit in clause):
            return False
    return True


In [ ]:
# --- Валидация самой редукции полным перебором на маленьких случайных примерах ---
# (Это НЕ решатель SAT и НЕ используется PDNAR -- только проверка, что редукция верна.)
import itertools


def brute_force_sat(clauses, n_vars):
    for bits in itertools.product([False, True], repeat=n_vars):
        if evaluate_cnf(clauses, list(bits)):
            return True, list(bits)
    return False, None


def brute_force_min_cover_size(num_x, num_y, edge_index):
    sets = [set() for _ in range(num_x)]
    for e in range(edge_index.shape[1]):
        y, x = edge_index[0, e].item(), edge_index[1, e].item()
        sets[x].add(y)
    universe = set(range(num_y))
    for r in range(1, num_x + 1):
        for combo in itertools.combinations(range(num_x), r):
            covered = set().union(*(sets[s] for s in combo)) if combo else set()
            if covered == universe:
                return r
    return None


rng = random.Random(1)
ok = True
for _ in range(25):
    nv = rng.randint(2, 4)
    ncl = rng.randint(2, 5)
    cls = []
    for _ in range(ncl):
        k = rng.randint(1, min(3, nv))
        vs = rng.sample(range(1, nv + 1), k)
        cls.append([v if rng.random() < 0.5 else -v for v in vs])
    is_sat, _ = brute_force_sat(cls, nv)
    data, meta = sat_to_bipartite_data(cls, nv)
    min_size = brute_force_min_cover_size(meta["num_x"], meta["num_y"], data.edge_index)
    predicted_sat = (min_size == nv)
    ok &= (predicted_sat == is_sat)

print("Редукция SAT -> Set Cover корректна на 25 случайных примерах:", ok)


Редукция SAT -> Set Cover корректна на 25 случайных примерах: True


## 3. Данные для обучения PDNAR

Генерируем случайные взвешенные экземпляры Set Cover (`generate_bipartite_graphs`,
как в `src/dataset/graph.py`) и размечаем их классическим взвешенным primal-dual
алгоритмом (`generate_set_cover_trace` = `set_cover(...)` из
`src/dataset/algorithms/set_cover.py`).

Этот алгоритм строит:
* пошаговую траекторию (`x`, `y`, `x_mask`, `y_mask`) -- обучающие hints,
* точное оптимальное покрытие `primal_optimal_solution` (через MILP, `scipy.optimize.milp`).

Это **ровно то же самое**, что делает `python main.py data.algorithm="set_cover"`
из README репозитория, только без Hydra/CLI, чтобы всё было в одном ноутбуке.
Число примеров и эпох ниже уменьшено для быстрой демонстрации -- для
воспроизведения результатов уровня статьи используйте официальный CLI с
дефолтными `n_train_samples=1000` и т.д. (см. `conf/data/data.yaml`).


In [ ]:
N_TRAIN_NODES = 16     # как дефолт в conf/data/data.yaml
N_TRAIN_SAMPLES = 250  # уменьшено для скорости демо (дефолт в репо -- 1000)
N_VAL_SAMPLES = 50
N_TEST_SAMPLES = 50
BATCH_SIZE = 32

print("Генерация случайных двудольных графов...")
train_graphs = generate_bipartite_graphs(N_TRAIN_SAMPLES, N_TRAIN_NODES)
val_graphs = generate_bipartite_graphs(N_VAL_SAMPLES, N_TRAIN_NODES)
test_graphs = generate_bipartite_graphs(N_TEST_SAMPLES, N_TRAIN_NODES)

print("Разметка классическим primal-dual алгоритмом (ТОЛЬКО для обучающих меток)...")
train_data = [generate_set_cover_trace(g) for g in tqdm(train_graphs, desc="train")]
val_data = [generate_set_cover_trace(g) for g in tqdm(val_graphs, desc="val")]
test_data = [generate_set_cover_trace(g) for g in tqdm(test_graphs, desc="test")]

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, follow_batch=["x", "y"])
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, follow_batch=["x", "y"])
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, follow_batch=["x", "y"])
print("Готово:", len(train_data), "train /", len(val_data), "val /", len(test_data), "test графов")


Генерация случайных двудольных графов...
Разметка классическим primal-dual алгоритмом (ТОЛЬКО для обучающих меток)...


train:   0%|          | 0/250 [00:00<?, ?it/s]

val:   0%|          | 0/50 [00:00<?, ?it/s]

test:   0%|          | 0/50 [00:00<?, ?it/s]

Готово: 250 train / 50 val / 50 test графов


## 4. Обучение PDNAR

Конфигурация модели и оптимизатора взята из `conf/model/model.yaml`
(`hidden_dim=32`, `eps=False` -- как для `set_cover`, см. README: команда для MSC
не включает `model.model.eps=True`, в отличие от Hitting Set). Обучаем через
`VCEPLightning` + `lightning.Trainer`, как в `main.py`, но без Hydra.


In [ ]:
model_cfg = SimpleNamespace(
    model=SimpleNamespace(hidden_dim=32, eps=False),
    train=SimpleNamespace(
        optimizer=SimpleNamespace(lr=1e-3, weight_decay=1e-4, beta1=0.9, beta2=0.999),
        scheduler=SimpleNamespace(factor=0.1, patience=10),
    ),
)

lit_model = VCEPLightning(model=model_cfg.model, train=model_cfg.train)
pdnar_model = lit_model.model  # это и есть GraphNeuralExecutor (PDNAR)
print(pdnar_model)


GraphNeuralExecutor(
  (x_encoder): Sequential(
    (0): Linear(in_features=1, out_features=32, bias=True)
    (1): ReLU()
  )
  (degree_encoder): Sequential(
    (0): Linear(in_features=1, out_features=32, bias=True)
    (1): ReLU()
  )
  (processor): BipartiteMPNN(
    (msg_x_to_y): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=32, bias=True)
      (3): ReLU()
    )
    (update_x): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=32, bias=True)
      (3): ReLU()
    )
    (global_pool): MinAggregation()
  )
  (x_del_decoder): Sequential(
    (0): Linear(in_features=32, out_features=1, bias=True)
  )
  (x_decoder): Sequential(
    (0): Linear(in_features=32, out_features=1, bias=True)
    (1): ReLU()
  )
  (y_decoder): Sequential(
    (0): Linear(in_features=32, out_features=1, bias=True)
    (1): ReLU()


In [ ]:
MAX_EPOCHS = 20  # для полноценного обучения увеличьте (или запустите main.py напрямую)

trainer = L.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="auto",
    devices=1,
    logger=False,
    enable_checkpointing=False,
)
trainer.fit(lit_model, train_dataloaders=train_loader, val_dataloaders=val_loader)


INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ GraphNeuralExecutor │ 12.7 K │ train │     0 │
└───┴───────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 12.7 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 12.7 K                                                                                               
Total estimated model params size (MB): 0.051                                                                      
Modules in train mode: 30                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: `Trainer.fit` stopped: `max_epochs=20` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=20` reached.


Оценим обученную сеть штатным способом репозитория -- методом
`GraphNeuralExecutor.test(...)`, который сравнивает предсказанное покрытие с
точным MILP-оптимумом на отложенных тестовых графах (это **тот же метод**,
который вызывается `main.py` в строке `trainer.test(model, ...)`).

In [ ]:
test_metrics = trainer.test(lit_model, dataloaders=test_loader)
test_metrics


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    test_final_set_acc     │     0.942307710647583     │
│  test_final_weight_ratio  │    1.0378916263580322     │
│   test_optimal_set_acc    │    0.8509615659713745     │
│ test_optimal_weight_ratio │    1.4542242288589478     │
└───────────────────────────┴───────────────────────────┘

[{'test_final_set_acc': 0.942307710647583,
  'test_final_weight_ratio': 1.0378916263580322,
  'test_optimal_weight_ratio': 1.4542242288589478,
  'test_optimal_set_acc': 0.8509615659713745}]

## 5. Инференс на одном SAT-примере

Берём одну конкретную формулу, сводим её к Set Cover (раздел 2) и прогоняем
**обученную сеть PDNAR** в автономном (`eval`) режиме -- ровно тот же вычислительный
процесс (encoder → `BipartiteMPNN` processor → decoder, авторегрессивно по шагам,
плюс лёгкий "safety net" из `GraphNeuralExecutor.test(...)` на случай, если сеть
сама не построила валидное покрытие), что и в оригинальном `test()`. Мы лишь
возвращаем само предсказанное покрытие, а не accuracy-метрики -- потому что для
нового экземпляра (в отличие от синтетических тестовых графов выше) заранее
неизвестного оптимума у нас нет.

**Классический primal-dual алгоритм здесь не вызывается ни разу.**


In [ ]:
n_vars = 6
clauses = [
    [1, -2, 3],
    [-1, 2, 4],
    [-3, -4, 5],
    [2, -5, 6],
    [-1, -2, -3],
    [4, 5, -6],
    [-4, 1, 3],
    [-6, -5, 2],
    [1, 6, -3],
]

print(f"Формула: {n_vars} переменных, {len(clauses)} дизъюнктов")
for c in clauses:
    lits = " \u2228 ".join(f"x{abs(l)}" if l > 0 else f"\u00acx{abs(l)}" for l in c)
    print("  (", lits, ")")

sat_data, meta = sat_to_bipartite_data(clauses, n_vars)
sat_loader = DataLoader([sat_data], batch_size=1, follow_batch=["x", "y"])
sat_batch = next(iter(sat_loader)).to(DEVICE)
print(meta)


Формула: 6 переменных, 9 дизъюнктов
  ( x1 ∨ ¬x2 ∨ x3 )
  ( ¬x1 ∨ x2 ∨ x4 )
  ( ¬x3 ∨ ¬x4 ∨ x5 )
  ( x2 ∨ ¬x5 ∨ x6 )
  ( ¬x1 ∨ ¬x2 ∨ ¬x3 )
  ( x4 ∨ x5 ∨ ¬x6 )
  ( ¬x4 ∨ x1 ∨ x3 )
  ( ¬x6 ∨ ¬x5 ∨ x2 )
  ( x1 ∨ x6 ∨ ¬x3 )
{'n_vars': 6, 'n_clauses': 9, 'num_x': 12, 'num_y': 15}


In [ ]:
@torch.no_grad()
def pdnar_predict(model, batch):
    # Инференс PDNAR на новом экземпляре без известного оптимума.
    #
    # Повторяет ТОЧНО ТУ ЖЕ логику, что и GraphNeuralExecutor.test(...)
    # (src/model/graph_executor.py): авторегрессионный прогон обученной сети
    # (encoder -> BipartiteMPNN processor -> decoder) с итеративным удалением
    # покрытых элементов, плюс идентичный "safety net" (жадное докрытие) из
    # оригинального test(), если сеть сама не построила валидное покрытие.
    # В отличие от .test(), возвращает само покрытие pred_set, а не метрики
    # относительно (неизвестного здесь) оптимума.
    model.eval()
    num_x = batch.x.shape[0]
    num_y = batch.y.shape[0]
    timesteps = batch.x.shape[1]
    edge_index = batch.edge_index
    x_weights = batch.x[:, 0]
    batch_index_x = batch.x_batch
    batch_index_y = batch.y_batch
    batch_size = torch.max(batch_index_x).item() + 1

    x_pred = batch.x[:, 0]
    pred_set = torch.zeros(num_x).unsqueeze(-1).to(batch.x.device)
    pred_y_mask = torch.tensor([True for _ in range(num_y)]).unsqueeze(-1).to(batch.x.device)
    graph_x_finishes = torch.zeros((num_x,), device=batch.x.device).bool()

    steps_used = 0
    for _ in range(timesteps):
        steps_used += 1
        h_x = model.x_encoder(x_pred)

        pred_x_mask = (1 - pred_set).bool()
        edge_mask = (pred_x_mask[edge_index[1]] & pred_y_mask[edge_index[0]]).squeeze(-1)
        if edge_mask.sum() == 0:
            break

        x_degree = degree(edge_index[:, edge_mask][1], num_x).unsqueeze(-1)
        x_degree = torch.log(x_degree + 1)
        h_x_degree = model.degree_encoder(x_degree)

        next_x, _, _ = model.processor(
            h_x=h_x, h_x_degree=h_x_degree, edge_index=edge_index,
            x_mask=pred_x_mask, y_mask=pred_y_mask, edge_mask=edge_mask,
            batch_index_x=batch_index_x, batch_index_y=batch_index_y,
            batch_size=batch_size, eps=model.eps,
        )

        x_del_pred = torch.sigmoid(model.x_del_decoder(next_x))
        x_pred = model.x_decoder(next_x)

        x_del_pred[pred_set.squeeze(-1) == 1] = 0.0
        delete_x_mask = (x_del_pred > 0.5).squeeze(-1)
        delete_x_mask = delete_x_mask & (~pred_set.squeeze(-1).bool()) & (~graph_x_finishes)
        pred_set[delete_x_mask] = 1

        delete_x_indices = torch.nonzero(delete_x_mask).squeeze(-1)
        deleted_edges_mask = torch.isin(edge_index[1], delete_x_indices)
        delete_y_indices = edge_index[0][deleted_edges_mask].unique()
        pred_y_mask[delete_y_indices] = False

        graph_finishes = torch.zeros((batch_size,), device=x_del_pred.device)
        graph_finishes = graph_finishes.scatter_reduce_(
            dim=0, index=batch_index_y, src=pred_y_mask.squeeze(-1).float(),
            reduce="amax", include_self=False,
        )
        graph_finishes = ~(graph_finishes.bool())
        graph_x_finishes = graph_finishes[batch_index_x]

        if not torch.any(pred_y_mask):
            break

    used_fallback = False
    if torch.any(pred_y_mask):
        used_fallback = True
        while torch.any(pred_y_mask):
            pred_x_mask = (1 - pred_set).bool()
            edge_mask = (pred_x_mask[edge_index[1]] & pred_y_mask[edge_index[0]]).squeeze(-1)
            x_degree = degree(edge_index[:, edge_mask][1], num_x).unsqueeze(-1)
            x_indices = torch.arange(num_x, device=batch.x.device).unsqueeze(-1)
            keep = pred_x_mask & ~graph_x_finishes.unsqueeze(-1)
            x_indices_k = x_indices[keep]
            x_remain_weights = x_weights[keep]
            x_degree_k = x_degree[keep]
            x_remain_weights_degree = x_remain_weights / x_degree_k
            min_idx = torch.argmin(x_remain_weights_degree)
            min_x = x_indices_k[min_idx]
            pred_set[min_x] = 1
            deleted_edges_mask = torch.isin(edge_index[1], torch.tensor([min_x], device=batch.x.device))
            delete_y_indices = edge_index[0][deleted_edges_mask].unique()
            pred_y_mask[delete_y_indices] = False

    return pred_set.squeeze(-1).long().cpu(), {"steps_used": steps_used, "used_fallback": used_fallback}


In [ ]:
pred_set, info = pdnar_predict(pdnar_model, sat_batch)
chosen_sets = pred_set.nonzero().squeeze(-1).tolist()

print("PDNAR выбрал X-узлы (индексы литеральных множеств):", chosen_sets)
print("Размер покрытия, найденного PDNAR:", int(pred_set.sum().item()), "  (нижняя граница = n_vars =", n_vars, ")")
print("Диагностика:", info)

assignment, ambiguous, uncovered = decode_assignment(pred_set, n_vars)
print("\nВосстановленное присваивание:")
for i, v in enumerate(assignment):
    print(f"  x{i+1} = {v}")
if ambiguous:
    print("Переменные, покрытые обоими литералами (неоднозначно):", ambiguous)
if uncovered:
    print("ВНИМАНИЕ, непокрытые переменные (в норме не должно происходить):", uncovered)

is_valid = evaluate_cnf(clauses, assignment)
print("\nПрисваивание удовлетворяет всем дизъюнктам формулы:", is_valid)
print("Вывод (по решению PDNAR): формула",
      "ВЫПОЛНИМА (SAT)" if is_valid else "не подтверждена как выполнимая этим присваиванием")


PDNAR выбрал X-узлы (индексы литеральных множеств): [0, 2, 5, 6, 8, 11]
Размер покрытия, найденного PDNAR: 6   (нижняя граница = n_vars = 6 )
Диагностика: {'steps_used': 3, 'used_fallback': False}

Восстановленное присваивание:
  x1 = True
  x2 = True
  x3 = False
  x4 = True
  x5 = True
  x6 = False

Присваивание удовлетворяет всем дизъюнктам формулы: True
Вывод (по решению PDNAR): формула ВЫПОЛНИМА (SAT)


### Проверка (только для контроля качества, не часть решения PDNAR)

Ниже -- полный перебор по SAT (годится только для маленьких `n_vars`) и точный
MILP-оптимум для того же экземпляра Set Cover (через `solve_minimum_set_cover`
из репозитория). Это исключительно валидация результата PDNAR, а не альтернативный
решатель, который используется вместо него.

In [ ]:
ground_truth_sat, ground_truth_assignment = brute_force_sat(clauses, n_vars)
print("Точный ответ (полный перебор):", "SAT" if ground_truth_sat else "UNSAT")
if ground_truth_sat:
    print("  witness:", {f"x{i+1}": v for i, v in enumerate(ground_truth_assignment)})
print("Совпадает с выводом PDNAR:", ground_truth_sat == is_valid)

num_x, num_y = meta["num_x"], meta["num_y"]
edges_per_y = [[] for _ in range(num_y)]
ei = sat_data.edge_index
for e in range(ei.shape[1]):
    y, x = ei[0, e].item(), ei[1, e].item()
    edges_per_y[y].append(x)
weights = [1.0] * num_x
opt_solution, opt_weight = solve_minimum_set_cover(num_x, num_y, weights, edges_per_y)
opt_size = int(round(sum(opt_solution)))
print("\nТочный минимальный размер покрытия (MILP):", opt_size)
print("Размер покрытия PDNAR:                      ", int(pred_set.sum().item()))


Точный ответ (полный перебор): SAT
  witness: {'x1': False, 'x2': False, 'x3': False, 'x4': False, 'x5': False, 'x6': False}
Совпадает с выводом PDNAR: True

Точный минимальный размер покрытия (MILP): 6
Размер покрытия PDNAR:                       6


## Итог

* Раздел 2 сводит произвольную КНФ-формулу к экземпляру Set Cover с гарантией:
  минимальное покрытие размера $n$ $\Leftrightarrow$ формула выполнима.
* Разделы 3-4 обучают **PDNAR** (`GraphNeuralExecutor` + `BipartiteMPNN`) на
  синтетических взвешенных экземплярах Set Cover, используя классический
  primal-dual алгоритм только как источник обучающих меток (teacher forcing),
  как и предполагает архитектура из репозитория.
* Раздел 5 запускает **обученную сеть** (не классический алгоритм) на графе,
  полученном из SAT-формулы, и по предсказанному покрытию восстанавливает
  присваивание переменных, которое затем проверяется прямой подстановкой в
  формулу.
